# Deep learning on text

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [3]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_text import preprocess_features
from src.preprocessing.pipelines.deep_learning import load_preprocessors
from src.models.on_text.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-06 17:05:39.173033: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 17:05:39.206990: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 17:05:40.065410: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_text)
importlib.reload(src.models.on_text.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_X_train=X_train
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
modality = 'text'
version = 1
artifacts_folder = Path(f'artifacts/on_text/deep_learning/v{version}')
image_artifacts_folder = Path(f'artifacts/on_images/deep_learning/v1')
log_file_path = image_artifacts_folder / 'experiments.parquet'
preprocessors_folder = image_artifacts_folder
tensor_board_folder = image_artifacts_folder / "tensorboard_logs"

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = False  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False


## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

In [13]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

In [14]:
y_train.value_counts().describe()

count      27.000000
mean     2516.000000
std      1682.489662
min       611.000000
25%      1241.000000
50%      2137.000000
75%      3813.500000
max      8167.000000
Name: count, dtype: float64

In [15]:
print(X_train.shape)

(67932, 31)


In [16]:
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target'],artifacts_folder=preprocessors_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib


In [17]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_X_train=full_X_train, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights)
preprocessors |= new_preprocessors

I0000 00:00:1759763141.433445   10328 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [18]:
new_preprocessors

{'text_vectorizer': <TextVectorization name=text_vectorization, built=False>}

In [19]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [20]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False)

## Model

In [21]:
from tensorflow import keras

### Load or create model

In [22]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    arch_version = last_experiment.get('arch_version', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    arch_version = last_experiment.get('arch_version', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(text_vectorizer=preprocessors['text_vectorizer'], num_classes=27)

Création d'un nouveau modèle.


In [23]:
if not load_model:
    arch_version = int(input(f"arch_version? (last: {arch_version})"))

In [24]:
arch_version

10

### Summary

In [25]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 310)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_embedding (Embedding)      │ (None, 310, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 306, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 27)             │         1,755 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,652,059 (10.12 MB)

 Trainable params: 2,652,059 (10.12 MB)

 Non-trainable params: 0 (0.00 B)

## Callbacks

### ModelCheckpoint

In [ ]:
# # Pick an available filename to save a model.
# arch_version=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{arch_version}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     arch_version+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{arch_version}.h5')
# new_location_for_saving_model


In [27]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [28]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [29]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [30]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_accuracy', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='max',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [31]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [32]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [33]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / f"{timestamp}-{modality}",
    histogram_freq=1  # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [42]:
import math
typical_minutes_per_epoch=1.2 * X_train.shape[0] / 67932

In [ ]:
# max_epochs=5

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

6

In [45]:
# Pick max_epochs based on your available time
available_minutes=30

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

25

### compilation and callbacks

In [46]:
learning_rate=0.001

In [47]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [48]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [49]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=30, max_epochs=25, champion_path=None ?

In [50]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=total_epochs_trained, callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/25


2025-10-06 17:06:53.581620: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


2123/2123 ━━━━━━━━━━━━━━━━━━━━ 31s 14ms/step - accuracy: 0.6100 - loss: 1.4043 - val_accuracy: 0.7792 - val_loss: 0.7889 - learning_rate: 0.0010
Epoch 2/25
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 26s 12ms/step - accuracy: 0.7970 - loss: 0.7408 - val_accuracy: 0.8076 - val_loss: 0.6849 - learning_rate: 0.0010
Epoch 3/25
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - accuracy: 0.8539 - loss: 0.5299 - val_accuracy: 0.8123 - val_loss: 0.7080 - learning_rate: 0.0010
Epoch 4/25
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.8925 - loss: 0.3947 - val_accuracy: 0.8171 - val_loss: 0.7425 - learning_rate: 0.0010
Epoch 5/25
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9199 - loss: 0.3045 - val_accuracy: 0.8149 - val_loss: 0.8229 - learning_rate: 0.0010
Epoch 6/25
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9326 - loss: 0.2580
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9372 - 

'total_minutes=6.997604330380757'

## Evaluation

In [51]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_text/deep_learning/v1/tensorboard_logs'


In [52]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 18,
 'minutes_per_epoch': 0.3887557961322643}

In [53]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [54]:
#Takes 1m40
y_pred = model.predict(test_ds)

531/531 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [55]:
from sklearn import metrics

In [56]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [57]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1180,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,,
10,340,32,1,0,5,8,1,6,7,4,0,1,10,0,0,1,0,2,39,81,0,4,1,0,2,78,0
40,25,340,19,2,13,7,5,16,14,0,0,3,2,0,0,0,4,3,3,8,24,2,0,1,3,8,0
50,1,14,270,4,3,0,0,1,4,3,1,3,0,2,0,1,3,0,0,0,17,0,3,0,5,1,0
60,0,6,9,134,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,0,1,0
1140,10,9,6,0,395,9,7,41,7,2,0,2,6,0,0,1,2,1,7,17,5,4,0,0,1,2,0
1160,10,6,1,0,4,738,2,5,3,1,0,0,1,0,0,0,0,2,5,9,1,1,0,0,0,2,0
1180,7,7,1,0,9,2,81,8,14,0,0,0,0,0,0,0,2,1,3,6,1,3,1,0,1,6,0
1280,3,4,2,0,36,2,4,674,51,101,4,21,21,2,2,0,14,4,1,5,0,11,5,2,1,4,0
1281,17,7,2,0,7,10,9,94,233,0,1,8,2,0,0,1,2,0,0,1,3,5,1,2,0,9,0


In [58]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.820939752935513, 0.0)

In [59]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [60]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.561056,0.545746,0.553295,623.000000
40,0.718816,0.677291,0.697436,502.000000
50,0.764873,0.803571,0.783745,336.000000
60,0.911565,0.807229,0.856230,166.000000
1140,0.766990,0.739700,0.753098,534.000000
1160,0.921348,0.932996,0.927136,791.000000
1180,0.637795,0.529412,0.578571,153.000000
1280,0.662083,0.691992,0.676707,974.000000
1281,0.619681,0.562802,0.589873,414.000000
1300,0.871094,0.884044,0.877521,1009.000000


In [61]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.807743,0.795654,0.801079,629.037037
std,0.111880,0.120413,0.114840,420.759339
min,0.561056,0.529412,0.553295,153.000000
25%,0.740925,0.750758,0.750595,310.000000
50%,0.828203,0.799383,0.798151,534.000000
75%,0.912406,0.879910,0.887805,953.500000
max,1.000000,0.982759,0.991304,2042.000000


In [62]:
# positive correlation between support and another measure can suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.939380,0.982211,0.031772
recall,0.939380,1.000000,0.987025,0.056831
f1-score,0.982211,0.987025,1.000000,0.046006
support,0.031772,0.056831,0.046006,1.000000


In [63]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.820939752935513

In [64]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 18,
 'minutes_per_epoch': 0.3887557961322643,
 'weighted_avg_f1_score': 0.820939752935513,
 'min_f1_score': np.float64(0.5532953620829943),
 'std_f1_score': np.float64(0.11483953865476623)}

## Update tracker

In [73]:
tracker['comment']="Performance better than the benchmark."
tracker['comment']

'Performance better than the benchmark.'

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmax(model_history.history['val_accuracy'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_accuracy = model_history.history['val_accuracy'][best_epoch_in_session_idx]
    tracker['val_accuracy'] = best_val_accuracy

    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_arch-{arch_version}_epoch_index-{best_epoch_global:02d}_val_accuracy-{best_val_accuracy:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_path)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print(f"Effacement de l'ancien modèle chargé {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le modèle chargé.")
    keep_candidate=False


Sauvegarde du modèle.
artifacts/on_text/deep_learning/v1/best_model_arch-10_epoch_index-12_val_accuracy-0.8214_f1-0.8209.keras


In [67]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

In [68]:
to_track=['arch_version','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp','modality']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model=None)

In [69]:
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 18,
 'minutes_per_epoch': 0.3887557961322643,
 'weighted_avg_f1_score': 0.820939752935513,
 'min_f1_score': np.float64(0.5532953620829943),
 'std_f1_score': np.float64(0.11483953865476623),
 'comment': 'Performance already better than on images.',
 'val_accuracy': 0.8213613033294678,
 'best_model_path': 'artifacts/on_text/deep_learning/v1/best_model_arch-10_epoch_index-12_val_accuracy-0.8214_f1-0.8209.keras',
 'epoch_index': np.int64(12),
 'total_epochs': np.int64(13),
 'arch_version': 10,
 'max_epochs': 25,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'timestamp': '20251006-170548',
 'modality': 'text',
 'dense_layers_sizes': {'dense': 64},
 'embedding_dims': {'text_embedding': 128}}

In [70]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [71]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [72]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [74]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

Log pour l'expérience arch_version 10 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [75]:
pd.set_option('max_colwidth', None)

In [76]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,modality,arch_version,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,timestamp,comment,best_model_path
0,image,2,False,6793,32,1.998601,10,0.001,0.569713,0.528593,0.000000,0.247884,256_128_64_32,16_16,None,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
1,image,2,False,20380,32,3.867331,8,0.001,0.584609,0.563841,0.000000,0.239801,256_128_64_32,16_16,None,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras
2,image,3,False,6793,32,2.024143,10,0.001,0.566710,0.527659,0.000000,0.243439,128_64_32_16,8_8,None,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
3,image,4,True,6793,32,1.954997,13,0.001,0.541922,0.490818,0.000000,0.270646,128_64_32_16,8_8,None,"Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.",artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras
4,image,4,True,20380,32,3.115025,4,0.001,0.562529,0.499731,0.000000,0.276383,128_64_32_16,8_8,None,Increased frac from .1 to .3.,artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-03_val_loss-1.7963_f1-0.4997.keras
5,image,5,True,20380,32,3.080263,3,0.001,0.566945,0.506920,0.000000,0.255305,128_64_32_16,8_8,None,Removed dropout layer before softmax. Performance better but more overfitting.,artifacts/on_images/deep_learning/v1/best_model_sv-5_epoch_index-02_val_loss-1.7414_f1-0.5069.keras
6,image,6,True,6793,32,1.781595,6,0.001,0.558585,0.537182,0.000000,0.223538,256_128_64_32,16_16,None,Frac back to .1. Reverted layers/embeddings to higher sizes and uncommented Dropout before softmax.,artifacts/on_images/deep_learning/v1/best_model_sv-6_epoch_index-05_val_loss-1.9153_f1-0.5372.keras
7,image,7,True,6793,16,2.911669,5,0.001,0.473976,0.410351,0.000000,0.261518,256_128_64_32,16_16,20251004-151611,Unfreezed base model. Batch size from 32 to 16 because memory error.,artifacts/on_images/deep_learning/v1/best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras
8,image,8,True,6793,32,1.819877,4,0.001,0.573775,0.523869,0.000000,0.235614,256_128_64_32,16_16,20251006-101001,"Unfreezed base model, so batch size back to 32. Lowered first dropout rate from .5 to .2.",artifacts/on_images/deep_learning/v1/best_model_sv-8_epoch_index-03_val_loss-1.9175_f1-0.5239.keras
9,image,9,True,67932,32,8.565354,3,0.001,0.591733,0.570463,0.012821,0.211199,256_128_64_32,16_16,20251006-130338,Full training set. Set callbacks to val_accuracy instead of val_loss.,artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-01_val_accuracy-0.5917_f1-0.5705.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
